<div align="center">
  <a href="https://colab.research.google.com/github/PrunaAI/ai-efficiency-courses/blob/main/solutions/06-use_data_llm_quantization.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>
</div>

---
**💡 Tip**: Click the button above to open this notebook in Google Colab for free GPU access!

## Installation

This notebook includes automatic setup cells that will install the project from git repository with UV.

**Note**: Run the setup cells below before starting the exercises.

In [ ]:
# Install project directly from git repository
!uv pip install git+https://github.com/PrunaAI/ai-efficiency-courses.git

# 07: Use Data in LLM Quantization

Welcome to this new lecture of the AI Efficiency course! 🚀

In this tutorial, we will explore data can be used during the quantization process. When data is available, it is reasonable to make use of it to improve the quantized models. The content from the chapter 4 [slides](slides/04-quantize_language_models.pdf) will help you to go through this notebook.

By the end of this lecture, you will:
- Understand how to use data to quantize LLMs.
- Evaluate the impact in the quantization of:
    - no data, 
    - in-distribution data, 
    - out-of-distribution data, 
    - synthetic data 

Let's get started on incorporating data during LLM quantization!

## 1. Imports

As we've already installed the project, we can import the necessary libraries. We will be using `torch` and `transformers` for this tutorial as interfaces to the model and tokenizer. On top of that, we will be using `matplotlib` for basic plotting. We recommend to checkout the [Pruna documentation](https://docs.pruna.ai/en/stable/docs_pruna/user_manual/evaluation.html) for access to AI efficiency functions.

Beyond external libraries, this course comes with the `course` local package which contains a lot of utils that you can use in the notebooks. 

Before starting, we highly recommend to check the basic [huggingface setup in the readme](https://github.com/PrunaAI/ai-efficiency-courses?tab=readme-ov-file#configuration) including updating cache directory, loging in to hugging face.

In [4]:
import gc
import copy
import random

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

from pruna_pro import SmashConfig
from pruna_pro import smash
from pruna.data.pruna_datamodule import PrunaDataModule
from pruna.data.utils import split_train_into_train_val_test
from pruna.evaluation.evaluation_agent import EvaluationAgent
from pruna.evaluation.metrics.metric_torch import TorchMetricWrapper
from pruna.evaluation.task import Task

Multiple distributions found for package optimum. Picked distribution: optimum


## 2. Utils

Similarly to other notebooks, we'll leverage some course utilities to streamline our workflow.
These utilities will help us:
- Load and manage lists of model ids that we have verified to work.
- Generate informative plots for model analysis.
- Iterate efficiently over evaluation and model configuration options.

Let's first load our models. We will use `SMALL_MODEL_IDS`, which are sub 1B parameters which should be easy to download and load into memory. We recommend starting with these smaller models but feel free to experiment with other models until you reach your GPU memory limit! 

In [1]:
from course import SMALL_MODEL_IDS as MODEL_IDS
# from course import MEDIUM_MODEL_IDS as MODEL_IDS
# from course import LARGE_MODEL_IDS as MODEL_IDS

MODEL_IDS

['facebook/opt-125m',
 'facebook/opt-350m',
 'HuggingFaceTB/SmolLM-135M-instruct',
 'HuggingFaceTB/SmolLM2-135M-Instruct',
 'HuggingFaceTB/SmolLM-360M-Instruct',
 'HuggingFaceTB/SmolLM2-360M-Instruct',
 'PleIAs/Pleias-350m-Preview',
 'PleIAs/Pleias-Pico',
 'LiquidAI/LFM2-350M',
 'LiquidAI/LFM2-700M']

## 3. Use data in LLM quantization

### 3.1 Evaluate the base model quality

**Why is this important?**
 Understanding baseline model performance helps establish a reference point for comparing
 quantized versions. Evaluating on different datasets and models provides insight into
 how quantization impacts vary across contexts.

**Your tasks:**
 - Evaluate the base model quality with the perplexity metric on the WikiText dataset
 - Repeat the experiment with other LLMs and/or datasets

**Key questions to answer as you explore:**
 - Check that the number are comparable to what you found in other notebooks, paper, repos.

In [8]:
from course.models import delete

def smash_evaluate_perplexity(model, tokenizer, smash_config, dataset="WikiText"):
    ### To Complete ###
    model_copy = copy.deepcopy(model)

    if smash_config:
        model_copy = smash(model_copy, smash_config)
    metrics = [TorchMetricWrapper(metric_name="perplexity", call_type="single")]
    task = Task(
        metrics, datamodule=PrunaDataModule.from_string(dataset, tokenizer=tokenizer)
    )
    eval_agent = EvaluationAgent(task)
    results = eval_agent.evaluate(model_copy)

    delete(model_copy)
    ### End of To Complete ###

    return results

In [6]:
### To Complete ###
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(MODEL_IDS[0]).cuda()
tokenizer = AutoTokenizer.from_pretrained(MODEL_IDS[0])

results = smash_evaluate_perplexity(model, tokenizer, None, dataset="WikiText")
print(results)
### End of To Complete ###

INFO - Using call_type: y_gt for metric perplexity
INFO - Testing compatibility with functools.partial(<function text_generation_collate at 0x7f967f63f490>, tokenizer=PreTrainedTokenizerFast(name_or_path='PleIAs/Pleias-350m-Preview', vocab_size=65536, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|end_of_text|>', 'eos_token': '<|end_of_text|>', 'unk_token': '[UNK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
), max_seq_len=None)...
INFO - Using

{'perplexity_y_gt': 34.41499710083008}


### 3.2 Quantize LLM without data

**Why is this important?**
Understanding how quantization without data affects model performance helps establish a baseline
for comparing more sophisticated quantization approaches. This provides insight into the value
of data-driven quantization methods.

**Your tasks:**
- Quantize the LLM with Quanto, which performs naive linear quantization without using any data
- Evaluate the quantized model quality using perplexity on WikiText dataset 
- Compare results to the base model performance

**Key questions to answer as you explore:**
- How much does performance degrade with naive quantization?
- What are the tradeoffs between model size reduction and quality loss?
- In what scenarios might data-free quantization be preferable?

In [9]:
### To Complete ###
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(MODEL_IDS[0]).cuda()
tokenizer = AutoTokenizer.from_pretrained(MODEL_IDS[0])

smash_config = SmashConfig()
smash_config.add_tokenizer(MODEL_IDS[0])
smash_config["quantizer"] = "quanto"
smash_config["quanto_weight_bits"] = "qint4"

results = smash_evaluate_perplexity(model, tokenizer, smash_config, dataset="WikiText")
print(results)
### End of To Complete ###

INFO - Using best available device: 'cuda'
INFO - Verifying Pruna token.
INFO - You have used 16055 hours this month.
INFO - Token was verified!
INFO - Starting quantizer quanto...
ERROR - Calibration requires a tokenizer and dataloader. Skipping calibration.
INFO - quantizer quanto was applied successfully.
INFO - You have used 16055 hours this month.
INFO - Token was verified!
INFO - Using best available device: 'cuda'
INFO - Using call_type: y_gt for metric perplexity
INFO - Using max_seq_len of tokenizer: None
INFO - Testing compatibility with text_generation_collate...
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
INFO - Using best available device: 'cuda'
INFO - Using provided list of metric instances.
INFO - Using best available device: 'cuda'
INFO - Evaluating a smashed model.
INFO - Detected transformers model. Using TransformerHandler.
- The first element of the batch is passed as i

[MetricResult(name='perplexity', params={'_defaults': {}, 'metric': Perplexity(), 'update_fn': <function default_update at 0x7f5412a9b740>, 'call_type': 'y_gt', 'metric_name': 'perplexity', 'higher_is_better': False}, result=65.99962615966797)]


### 3.3 Quantize LLM with in-distribution data

**Why is this important?**
Understanding how data-driven quantization affects model performance helps evaluate the benefits
of using in-distribution data during quantization. This provides insights into optimizing the
quantization process for better model quality.

**Your tasks:**
- Quantize the LLM with GPTQ using in-distribution data from WikiText
- Evaluate the quantized model quality using perplexity on WikiText dataset
- Compare results to data-free quantization performance

**Key questions to answer as you explore:**
- How much does in-distribution data improve quantization quality?
- What are the tradeoffs between data collection effort and quality gains?
- In what scenarios is data-driven quantization worth the additional complexity?

In [8]:
### To Complete ###
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(MODEL_IDS[0]).cuda()
tokenizer = AutoTokenizer.from_pretrained(MODEL_IDS[0])

smash_config = SmashConfig()
smash_config.add_tokenizer(MODEL_IDS[0])
smash_config.add_data("WikiText", tokenizer=tokenizer)
smash_config["quantizer"] = "gptq"
smash_config["gptq_weight_bits"] = 4

results = smash_evaluate_perplexity(model, tokenizer, smash_config, dataset="WikiText")
print(results)
### End of To Complete ###

INFO - Testing compatibility with functools.partial(<function text_generation_collate at 0x7f967f63f490>, tokenizer=PreTrainedTokenizerFast(name_or_path='PleIAs/Pleias-350m-Preview', vocab_size=65536, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|end_of_text|>', 'eos_token': '<|end_of_text|>', 'unk_token': '[UNK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
), max_seq_len=None)...
Asking to truncate to max_length but no maximum length is provi

Quantizing model.layers blocks :   0%|          | 0/26 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

...

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
INFO - quantizer gptq was applied successfully.
INFO - You have used 146 hours this month.
INFO - Using call_type: y_gt for metric perplexity
INFO - Testing compatibility with functools.partial(<function text_generation_collate at 0x7f967f63f490>, tokenizer=PreTrainedTokenizerFast(name_or_path='PleIAs/Pleias-350m-Preview', vocab_size=65536, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|end_of_text|>', 'eos_token': '<|end_of_text|>', 'unk_token': '[UNK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False

{'perplexity_y_gt': 37.480918884277344}


### 3.4 Quantize LLM with more/less in-distribution data

**Why is this important?**
Understanding how the amount of in-distribution data (i.e. data similar to the data met during inference) affects quantization helps optimize the data collection process.
This provides insights into the minimum data requirements needed for effective quantization.

**Your tasks:**
- Quantize the LLM with GPTQ or AWQ using a small/large subset of WikiText data
- Evaluate the quantized model quality using perplexity on WikiText dataset
- Compare results to quantization with full dataset

**Key questions to answer as you explore:**
- How much in-distribution data is needed for good quantization results?
- What is the relationship between data amount and model quality?
- At what point do additional data samples provide diminishing returns?

In [9]:
### To Complete ###
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(MODEL_IDS[0]).cuda()
tokenizer = AutoTokenizer.from_pretrained(MODEL_IDS[0])

train_ds, val_ds, test_ds = load_dataset(
    "mikasenghaas/wikitext-2", split=["train", "validation", "test"]
)
train_ds = train_ds.select(range(1000))

smash_config = SmashConfig()
smash_config.add_tokenizer(model_id)
smash_config.add_data((train_ds, val_ds, test_ds), collate_fn="text_generation_collate")
smash_config["quantizer"] = "gptq"
smash_config["gptq_weight_bits"] = 4

results = smash_evaluate_perplexity(model, tokenizer, smash_config, dataset="WikiText")
print(results)
### End of To Complete ###

INFO - Using max_seq_len of tokenizer: None
INFO - Testing compatibility with functools.partial(<function text_generation_collate at 0x7f967f63f490>, tokenizer=PreTrainedTokenizerFast(name_or_path='PleIAs/Pleias-350m-Preview', vocab_size=65536, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|end_of_text|>', 'eos_token': '<|end_of_text|>', 'unk_token': '[UNK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
), max_seq_len=None)...
Asking to truncate 

Quantizing model.layers blocks :   0%|          | 0/26 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

...

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

INFO - quantizer gptq was applied successfully.
INFO - You have used 147 hours this month.
INFO - Using call_type: y_gt for metric perplexity
INFO - Testing compatibility with functools.partial(<function text_generation_collate at 0x7f967f63f490>, tokenizer=PreTrainedTokenizerFast(name_or_path='PleIAs/Pleias-350m-Preview', vocab_size=65536, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|end_of_text|>', 'eos_token': '<|end_of_text|>', 'unk_token': '[UNK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[PAD]", rstrip=False, lstrip=False,

{'perplexity_y_gt': 37.480918884277344}


### 3.5 Quantize LLM with random data

**Why is this important?**
Understanding how random data affects quantization helps determine if data quality matters.
This provides insights into whether collecting high-quality in-distribution data is necessary.

**Your tasks:**
- Quantize the LLM with GPTQ or AWQ using randomly generated text data
- Evaluate the quantized model quality using perplexity on WikiText dataset
- Compare results to quantization with no data or real data

**Key questions to answer as you explore:**
- Does random data provide effective quantization?
- How does model quality compare to using no data or real text data?
- What are the implications for data collection requirements?

In [10]:
### To Complete ###
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(MODEL_IDS[0]).cuda()
tokenizer = AutoTokenizer.from_pretrained(MODEL_IDS[0])

train_ds = [
    {"text": "".join([chr(random.randint(97, 122)) for _ in range(100)])}
    for _ in range(1000)
]
val_ds = [
    {"text": "".join([chr(random.randint(97, 122)) for _ in range(100)])}
    for _ in range(100)
]
test_ds = [
    {"text": "".join([chr(random.randint(97, 122)) for _ in range(100)])}
    for _ in range(100)
]

smash_config = SmashConfig()
smash_config.add_tokenizer(model_id)
smash_config.add_data((train_ds, val_ds, test_ds), collate_fn="text_generation_collate")
smash_config["quantizer"] = "gptq"
smash_config["gptq_weight_bits"] = 4

results = smash_evaluate_perplexity(model, tokenizer, smash_config, dataset="WikiText")
print(results)
### End of To Complete ###

INFO - Testing compatibility with functools.partial(<function text_generation_collate at 0x7f967f63f490>, tokenizer=PreTrainedTokenizerFast(name_or_path='PleIAs/Pleias-350m-Preview', vocab_size=65536, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|end_of_text|>', 'eos_token': '<|end_of_text|>', 'unk_token': '[UNK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
), max_seq_len=None)...
Asking to truncate to max_length but no maximum length is provi

Quantizing model.layers blocks :   0%|          | 0/26 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

...

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

INFO - quantizer gptq was applied successfully.
INFO - You have used 147 hours this month.
INFO - Using call_type: y_gt for metric perplexity
INFO - Testing compatibility with functools.partial(<function text_generation_collate at 0x7f967f63f490>, tokenizer=PreTrainedTokenizerFast(name_or_path='PleIAs/Pleias-350m-Preview', vocab_size=65536, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|end_of_text|>', 'eos_token': '<|end_of_text|>', 'unk_token': '[UNK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[PAD]", rstrip=False, lstrip=False,

{'perplexity_y_gt': 42.67584228515625}


### 3.6 Quantize LLM with out-of-disitribution data

**Why is this important?**
Understanding how out-of-distribution data (i.e. data which is different from the data met during inference) affects quantization helps optimize data selection.
This provides insights into whether domain-specific data is needed for effective quantization.

**Your tasks:**
- Quantize the LLM with GPTQ or AWQ using BookCorpus dataset
- Evaluate the quantized model quality using perplexity on WikiText dataset
- Compare results to quantization with in-distribution data

**Key questions to answer as you explore:**
- How does out-of-distribution data affect quantization quality?
- What is the relationship between data domain and model quality?
- Is domain-specific data necessary for good quantization results?

In [ ]:
### To Complete ###
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(MODEL_IDS[0]).cuda()
tokenizer = AutoTokenizer.from_pretrained(MODEL_IDS[0])

train_ds = load_dataset("SamuelYang/bookcorpus")["train"]
train_ds, val_ds, test_ds = split_train_into_train_val_test(train_ds, seed=42)
train_ds = train_ds.select(range(1000))

model = model.to("cuda")
smash_config = SmashConfig()
smash_config.add_tokenizer(model_id)
smash_config.add_data((train_ds, val_ds, test_ds), collate_fn="text_generation_collate")
smash_config["quantizer"] = "gptq"
smash_config["gptq_weight_bits"] = 4

results = smash_evaluate_perplexity(model, tokenizer, smash_config, dataset="WikiText")
print(results)
### End of To Complete ###

INFO - Loaded only training, splitting train 80/10/10 into train, validation and test...
INFO - Using max_seq_len of tokenizer: None
INFO - Testing compatibility with functools.partial(<function text_generation_collate at 0x7f6b1dad3d90>, tokenizer=PreTrainedTokenizerFast(name_or_path='PleIAs/Pleias-350m-Preview', vocab_size=65536, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|end_of_text|>', 'eos_token': '<|end_of_text|>', 'unk_token': '[UNK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|begin_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|end_of_text|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[PAD]", rstrip=False, lstrip=False, single_w


INFO  ENV: Auto setting PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True' for memory saving.
INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


## Conclusion: What We've Learned About LLM on CPU and GPU

In this module, we explored quantizing LLMs on without data or with different types of data. Here are the key findings:

- **Real Data > Synthetic Data > No Data:**
  Using real data during quantization improved the performance of the quantized models. No data can still performs fairly good.

- **In-distribution Data > Out-of-distribution Data:**
  Using data close to data that will be used during inference to quantize a model improves performance of the quantized model.

- **More Data Helps a Bit:**
  While more data achieves better results, it is not strictly required to have huge datasets to achieve very good results

### Next Steps: Leverage Data During Fine-Tuning
 
Now that you understand how data affects quantization quality, you can make informed decisions about what data to use when quantizing models. The next sections will explore additional techniques to leverage data during fine-tuning (which can serve as recovery after quantization).
 
👉 **Continue to the next notebook:**
[07-finetune_llm.ipynb on GitHub](exercises/07-finetune_llm.ipynb). The content from the chapter 5 [slides](slides/05-finetuning_for_llms.pdf) will help you to go through this notebook.